> Copyright 2019 Google Inc.
>
> Código traduzido e adaptado de https://github.com/google-research

# Previsão de sentimento em críticas de filmes com BERT no TF Hub

Um dos modelos bem consolidados na área de processamento de linguagem natural (*natural language processing, NLP*) é o BERT (*bidirectional encoder representations from transformers*). Trata-se de uma arquitetura de rede neural desenvolvida por pesquisadores da Google que revolucionou o estado da arte em tarefas de NLP, como classificação de texto, tradução, sumarização e resposta a perguntas.

Agora que o BERT foi disponibilizado no Hugging Face como um módulo carregável, é relativamente fácil integrá-lo a *pipelines* de texto. Além disso, o ajuste fino (*fine-tunning*) do BERT pode proporcionar tanto um aumento na precisão dos resultados obtidos quanto em um tempo de treinamento do modelo mais reduzido.

A evolução dos modelos de processamento de linguagem natural parte das redes neurais recorrentes (RNNs), evoluindo para arquiteturas LSTM e Bi-LSTM, posteriormente combinadas com mecanismos de atenção e estruturas encoder-decoder. Essa trajetório culmina na arquitetura Transformer, que substitui a recorrência por mecanismos de atenção, permitindo maior paralelização e melhor modelagem das dependências de longo alcance. Assim, o modelo empregado nesta etapa representa um avanço significativo em relação às arquiteturas RNN, LSTM e Bi-LSTM apresentadas anteriormente. A Figura 1 a seguir apresenta, de forma simplificada, essa progressão arquitetural e posiciona o BERT no contexto dos modelos abordados ao longo dos experimentos.

<figure style="text-align: center;">
  <img src="assets/figura-4.png" alt="Figura 4" width="800">
</figure>

**Figura 1** – Evolução das arquiteturas de processamento de sequências até o BERT.

Aqui, treinaremos um modelo para prever se uma crítica de filme do IMDB é positiva ou negativa, usando o mesmo conjunto de dados que foi trabalhando nos últimos laboratórios, só que dessa vez utilizando a arquitetura do BERT.

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 1:</b><br>
  Qual é a principal diferença entre a abordagem utilizada pelo BERT e as arquiteturas recorrentes, como RNNs e LSTMs, apresentadas nos laboratórios anteriores? Por que o mecanismo de atenção é importante para o processamento de textos?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

In [1]:
# !python -m pip install keras-hub

In [2]:
import os
import re
import shutil
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras

import keras_hub
from transformers import BertTokenizer
from sklearn.manifold import TSNE
from sklearn.metrics import confusion_matrix

In [3]:
# diretório para salvar os arquivos do modelo
OUTPUT_DIR = Path("output")

# se True, apaga o diretório existente antes de criar um novo
DO_DELETE = False

if DO_DELETE and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Importando os Dados

Primeiro, vamos baixar o conjunto de dados hospedados pela Stanford, uma forma diferente até então de como o conjunto de dados foi importado nos outros laboratórios. O código a seguir baixa, extrai e importa o *IMDB Large Movie Review Dataset*.

In [4]:
# carrega todos os arquivos de um diretório em um DataFrame
def load_directory_data(directory):
    data = {}
    data["sentence"] = []
    data["sentiment"] = []

    for file_path in os.listdir(directory):
        with open(os.path.join(directory, file_path), "r", encoding="utf-8") as f:
            data["sentence"].append(f.read())

        data["sentiment"].append(re.match(r"\d+_(\d+)\.txt", file_path).group(1))

    return pd.DataFrame.from_dict(data)

# concatena exemplos positivos e negativos, adiciona uma coluna de polaridade e embaralha
def load_dataset(directory):
    pos_df = load_directory_data(os.path.join(directory, "pos"))

    neg_df = load_directory_data(os.path.join(directory, "neg"))

    pos_df["polarity"] = 1
    neg_df["polarity"] = 0

    return pd.concat(
        [pos_df, neg_df], ignore_index=True
    ).sample(frac=1,random_state=42).reset_index(drop=True)

# baixa e processa os arquivos em um diretório
def download_and_load_datasets(force_download=False):

    dataset = tf.keras.utils.get_file(
        fname="aclImdb.tar.gz",
        origin="https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz",
        extract=True
    )

    dataset_dir = os.path.join(os.path.dirname(dataset), "aclImdb_extracted", "aclImdb")

    train_df = load_dataset(os.path.join(dataset_dir, "train"))

    test_df = load_dataset(os.path.join(dataset_dir, "test"))

    return train_df, test_df

In [5]:
train, test = download_and_load_datasets()

84125825/84125825 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [6]:
# o dataset completo é usado para comparação justa com os outros modelos
print(f"Amostras de treino: {len(train)}")
print(f"Amostras de teste: {len(test)}")

Amostras de treino: 25000
Amostras de teste: 25000


In [7]:
train.columns

Index(['sentence', 'sentiment', 'polarity'], dtype='object')

Os dados de entrada são a coluna `sentence` e o rótulo é a coluna `polarity` (0 e 1 para negativo e positivo, respectivamente).

In [8]:
DATA_COLUMN = 'sentence'
LABEL_COLUMN = 'polarity'
# label_list é a lista de rótulos, isto é, True, False ou 0, 1
label_list = [0, 1]

## Processamento de Dados

Nesta etapa, os dados são separados em duas componentes: textos, que correspondem às resenhas utilizadas como entrada do modelo, e rótulos, que representam a polaridade de cada resenha. De modo que o texto possa ser posteriormente processado pelo tokenizador BERT, enquanto os rótulos são utilizados como variável alvo durante o treinamento supervisionado.

Os textos são convertidos para `str` e armazenados em listas Python por meio de `tolist()`. Já os rótulos são convertidos para o tipo inteiro `int32` e armazenados como *arrays* NumPy. A mesma operação é realizada para os conjuntos de treinamento e teste.

In [9]:
# separa textos e rótulos para treinamento e teste
train_texts = train[DATA_COLUMN].astype(str).tolist()
train_labels = train[LABEL_COLUMN].astype("int32").to_numpy()

test_texts = test[DATA_COLUMN].astype(str).tolist()
test_labels = test[LABEL_COLUMN].astype("int32").to_numpy()

print(f"Dados de treinamento: {len(train_texts)}")
print(f"Dados de teste: {len(test_texts)}")

Dados de treinamento: 25000
Dados de teste: 25000


Nesta etapa, é carregado o tokenizador pré-treinado do `BERT Base Uncased`. O texto de exemplo é então dividido em *tokens* utilizando o vocabulário e as regras de tokenização do BERT, incluindo a possibilidade de decompor palavras desconhecidas em *subword tokens*.

In [10]:
tokenizer = BertTokenizer.from_pretrained(
    "bert-base-uncased"
)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [11]:
text = "This here's an example of using the BERT tokenizer"

tokens = tokenizer.tokenize(text)

print(tokens)

['this', 'here', "'", 's', 'an', 'example', 'of', 'using', 'the', 'bert', 'token', '##izer']


As resenhas são convertidas em representações compatíveis com o BERT por meio do tokenizador previamente carregado. É definido um comprimento máximo de 256 tokens por sequência. Resenhas maiores são truncadas, enquanto sequências menores são preenchidas (*padding*) até esse tamanho.

O resultado é armazenado como *arrays* NumPy, contendo as informações necessárias para que o modelo consiga extrair os padrões, como os identificadores dos *tokens*, os identificadores de segmento e a máscara de atenção.

In [12]:
# define o tamanho máximo das sequências
MAX_SEQ_LENGTH = 256

# tokeniza as resenhas de treinamento
train_features = tokenizer(
    train_texts,
    max_length=MAX_SEQ_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="np"
)

# tokeniza as resenhas de teste
test_features = tokenizer(
    test_texts,
    max_length=MAX_SEQ_LENGTH,
    padding="max_length",
    truncation=True,
    return_tensors="np"
)

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 2:</b><br>
  Relembrando alguns detalhes discutidos em laboratórios anteriores, o que acontece com uma resenha que possui mais de 256 tokens? E com uma resenha que possui menos de 256 tokens? Quais são as possíveis consequências de truncar textos longos?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

## Criando o Modelo

<figure style="text-align: center;">
  <img src="assets/figura-5.png" alt="Figura 2" width="800">
</figure>

**Figura 2** – Arquitetura BERT para o problema de análise de sentimentos nas *reviews* do IMDB.

As três entradas produzidas pela tokenização (`padding_mask`, `segment_ids` e `token_ids`), restritas ao comprimento de 128 posições, alimentam diretamente o *backbone* pré-treinado do BERT.

Do ponto de vista estrutural, a visualização evidencia que o peso computacional da rede recai quase inteiramente no bloco do BERT, responsável por processar a sequência utilizando 109.482.240 parâmetros. O `pooled_output` resultante atua como um extrator de características, comprimindo o contexto semântico da resenha em um vetor de dimensão 768.

Após a aplicação do *dropout* com taxa de 0.1 como regularizador, a rede culmina em uma camada classificadora densa leve, contendo apenas 1.538 parâmetros. Essa camada é responsável por projetar as características diretamente nos dois *logits* de saída (para as classes Negativa e Positiva).

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 3:</b><br>
  Qual é a função de token_ids, segment_ids e padding_mask? Por que o modelo precisa receber uma máscara de atenção?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

In [13]:
# carrega o backbone bert-base-uncased pré-treinado
bert_backbone = keras_hub.models.BertBackbone.from_preset(
    "bert_base_en_uncased"
)

100%|██████████| 457/457 [00:00<00:00, 1.22MB/s]


100%|██████████| 418M/418M [00:09<00:00, 44.8MB/s]


Nesta etapa são definidos os principais hiperparâmetros utilizados no ajuste fino do BERT. O treinamento utiliza *batches* de 32 exemplos, taxa de aprendizado de $2.10^{-5}$ e 3 épocas.

Também é definida uma proporção de 10% dos passos de treinamento para *warmup*, estratégia na qual a taxa de aprendizado é gradualmente aumentada no início do treinamento. Por fim, como o problema consiste em classificação binária de sentimentos, são consideradas duas classes: negativa e positiva.

In [14]:
# hiperparâmetros de treinamento
BATCH_SIZE = 32
LEARNING_RATE = 2e-5
NUM_TRAIN_EPOCHS = 4

# proporção do treinamento utilizada para warmup
WARMUP_PROPORTION = 0.1

# número de classes
NUM_LABELS = 2

In [15]:
# número de passos de treinamento
num_train_steps = int(
    len(train_labels) / BATCH_SIZE * NUM_TRAIN_EPOCHS
)

# número de passos de warmup
num_warmup_steps = int(
    num_train_steps * WARMUP_PROPORTION
)

print("Train steps:", num_train_steps)
print("Warmup steps:", num_warmup_steps)

Train steps: 3125
Warmup steps: 312


A seguir, serão configurados *callbacks* para acompanhar e preservar os resultados do treinamento. O `ModelCheckpoint` salva uma cópia do modelo ao final de cada época, permitindo recuperar posteriormente o estado treinado sem necessidade de executar o treinamento novamente.

In [16]:
# salva apenas o melhor modelo (menor val_loss), não o da última época
checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath=str(OUTPUT_DIR / "bert_best.keras"),
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

# para o treino se val_loss não melhorar por 2 épocas e restaura o melhor pesos
early_stopping_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True,
    verbose=1
)

# callback para registrar métricas no tensorboard
tensorboard_callback = keras.callbacks.TensorBoard(
    log_dir=str(OUTPUT_DIR / "logs")
)

Nesta etapa é construída a arquitetura utilizada para a classificação de sentimentos. O modelo recebe três entradas produzidas durante a tokenização: `token_ids`, que identificam os tokens da sequência; `segment_ids`, que identificam os segmentos; e `padding_mask`, que indica quais posições correspondem a tokens reais e quais foram adicionadas por preenchimento.

As entradas são processadas pelo modelo BERT pré-treinado, a partir do qual é extraído o `pooled_output`, uma representação vetorial contextualizada da sequência. Essa representação é encaminhada para uma camada de *dropout* com taxa de 10%, utilizada como mecanismo de regularização, e posteriormente para uma camada `Dense` com duas saídas, correspondentes às classes de nosso interessse: Avaliação negativa ou positiva.

A função de perda utilizada é a `SparseCategoricalCrossentropy` com `from_logits=True`, adequada para classificação multiclasse quando a última camada fornece *logits* diretamente, sem aplicação prévia de `softmax`. O otimizador Adam utiliza a taxa de aprendizado definida anteriormente ($2.10^{-5}$), enquanto a acurácia é utilizada como métrica de avaliação durante o treinamento.

In [17]:
# número de classes
NUM_LABELS = len(label_list)

# entradas do modelo
inputs = {
    "token_ids": keras.Input(
        shape=(MAX_SEQ_LENGTH,),
        dtype="int32",
        name="token_ids"
    ),
    "segment_ids": keras.Input(
        shape=(MAX_SEQ_LENGTH,),
        dtype="int32",
        name="segment_ids"
    ),
    "padding_mask": keras.Input(
        shape=(MAX_SEQ_LENGTH,),
        dtype="bool",
        name="padding_mask"
    )
}

# passa as entradas pelo BERT
bert_outputs = bert_backbone(inputs)

# representação do token [CLS] após tanh (pooled output do bert)
pooled_output = bert_outputs["pooled_output"]

# dropout mais alto (0.3) regulariza melhor a camada de classificação que aprende sobre o BERT
x = keras.layers.Dropout(0.3)(pooled_output)

# camada intermediária com gelu (mesma ativação usada internamente pelo BERT)
x = keras.layers.Dense(256, activation="gelu", name="intermediate")(x)

# layer norm estabiliza as ativações antes do classificador final
x = keras.layers.LayerNormalization()(x)

# segundo dropout antes da camada de saída
x = keras.layers.Dropout(0.3)(x)

# camada de classificação final
outputs = keras.layers.Dense(
    NUM_LABELS,
    name="classifier"
)(x)

# modelo completo
model = keras.Model(
    inputs=inputs,
    outputs=outputs
)

model.summary()

# schedule com warmup linear seguido de decaimento linear
lr_decay = keras.optimizers.schedules.PolynomialDecay(
    initial_learning_rate=LEARNING_RATE,
    decay_steps=num_train_steps - num_warmup_steps,
    end_learning_rate=0.0,
    power=1.0
)

# schedule de warmup linear, sobe de 0 até LEARNING_RATE em num_warmup_steps
class LinearWarmupSchedule(keras.optimizers.schedules.LearningRateSchedule):
    # schedule de warmup linear seguido de decaimento
    def __init__(self, warmup_steps, decay_schedule, peak_lr):
        self.warmup_steps = tf.cast(warmup_steps, tf.float32)
        self.decay_schedule = decay_schedule
        self.peak_lr = peak_lr

    def __call__(self, step):
        step_f = tf.cast(step, tf.float32)
        warmup_lr = self.peak_lr * (step_f / self.warmup_steps)
        decay_lr = self.decay_schedule(step_f - self.warmup_steps)
        return tf.cond(
            step_f < self.warmup_steps,
            lambda: warmup_lr,
            lambda: decay_lr
        )

    def get_config(self):
        return {
            "warmup_steps": int(self.warmup_steps.numpy()),
            "peak_lr": self.peak_lr
        }

lr_schedule = LinearWarmupSchedule(
    warmup_steps=num_warmup_steps,
    decay_schedule=lr_decay,
    peak_lr=LEARNING_RATE
)

# adamw com weight decay e clipagem de gradiente, clipnorm=1.0 é o padrão para evitar explosão de gradientes
model.compile(
    optimizer=keras.optimizers.AdamW(
        learning_rate=lr_schedule,
        weight_decay=0.01,
        clipnorm=1.0
    ),
    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),
    metrics=["accuracy"]
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ padding_mask        │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ segment_ids         │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_ids           │ (None, 256)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bert_backbone       │ [(None, 768),     │ 109,482,2… │ padding_mask[0][… │
│ (BertBackbone)      │ (None, 256, 768)] │            │ segment_ids[0][0… │
│                     │                   │            │ token_ids[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_12          │ (None, 768)       │          0 │ bert_backbone[0]… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ intermediate        │ (None, 256)       │    196,864 │ dropout_12[0][0]  │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 256)       │        512 │ intermediate[0][… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 256)       │          0 │ layer_normalizat… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ classifier (Dense)  │ (None, 2)         │        514 │ dropout_13[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 109,680,130 (418.40 MB)

 Trainable params: 109,680,130 (418.40 MB)

 Non-trainable params: 0 (0.00 B)

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 4:</b><br>
  O BERT possui mais de 109 milhões de parâmetros, enquanto a camada de classificação possui uma quantidade muito menor. O que isso indica sobre onde está concentrada a capacidade de representação do modelo?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>



As estruturas produzidas pelo tokenizador Hugging Face utilizam nomes de campos diferentes dos esperados pelo `BertBackbone` do KerasHub. Então, é necessário realizar o mapeamento entre essas representações.

| Hugging Face | KerasHub | Função |
|---|---|---|
| `input_ids` | `token_ids` | Identificadores dos tokens |
| `token_type_ids` | `segment_ids` | Identificadores dos segmentos |
| `attention_mask` | `padding_mask` | Indica tokens válidos e posições de preenchimento |

In [18]:
# converte as saídas do tokenizer Hugging Face para os nomes esperados pelo KerasHub.
train_x = {
    "token_ids": train_features["input_ids"],
    "segment_ids": train_features["token_type_ids"],
    "padding_mask": train_features["attention_mask"].astype(bool)
}

test_x = {
    "token_ids": test_features["input_ids"],
    "segment_ids": test_features["token_type_ids"],
    "padding_mask": test_features["attention_mask"].astype(bool)
}

print("Train:")
for key, value in train_x.items():
    print(f"{key}: {value.shape}")

print("\nTest:")
for key, value in test_x.items():
    print(f"{key}: {value.shape}")

Train:
token_ids: (25000, 256)
segment_ids: (25000, 256)
padding_mask: (25000, 256)

Test:
token_ids: (25000, 256)
segment_ids: (25000, 256)
padding_mask: (25000, 256)


Nesta etapa, os dados de entrada e seus respectivos rótulos são organizados em um `tf.data.Dataset`, estrutura utilizada pelo TensorFlow para fornecer os exemplos ao modelo durante o treinamento.

Os exemplos são inicialmente embaralhados (`shuffle`) para reduzir possíveis efeitos relacionados à ordem dos dados. Em seguida, são agrupados em *batches* de tamanho definido por `BATCH_SIZE`. O parâmetro `drop_remainder=False` garante que, caso o número total de exemplos não seja divisível pelo tamanho do *batch*, o último grupo de exemplos também seja utilizado.

Por fim, `prefetch(tf.data.AUTOTUNE)` permite que o TensorFlow prepare os próximos *batches* enquanto o modelo processa o atual.

In [19]:
# split de validação: 10% do conjunto de treino (2.500 amostras)
# necessário para que early stopping e ModelCheckpoint funcionem
val_ratio = 0.1
val_size = int(val_ratio * len(train_labels))

# fatia de validação (primeiros val_size exemplos)
val_x = {k: v[:val_size] for k, v in train_x.items()}
val_labels_split = train_labels[:val_size]

# fatia de treino real (resto)
train_x_fit = {k: v[val_size:] for k, v in train_x.items()}
train_labels_fit = train_labels[val_size:]

print(f"Amostras de treino real: {len(train_labels_fit)}")
print(f"Amostras de validação: {len(val_labels_split)}")

# dataset de treinamento
train_dataset = tf.data.Dataset.from_tensor_slices(
    (train_x_fit, train_labels_fit)
)

train_dataset = train_dataset.shuffle(
    buffer_size=len(train_labels_fit)
)

train_dataset = train_dataset.batch(
    BATCH_SIZE,
    drop_remainder=False
)

train_dataset = train_dataset.prefetch(
    tf.data.AUTOTUNE
)

# dataset de validação (sem shuffle)
val_dataset = tf.data.Dataset.from_tensor_slices(
    (val_x, val_labels_split)
)

val_dataset = val_dataset.batch(
    BATCH_SIZE,
    drop_remainder=False
)

val_dataset = val_dataset.prefetch(
    tf.data.AUTOTUNE
)

Amostras de treino real: 22500
Amostras de validação: 2500


Como o ajuste fino (*fine-tuning*) do BERT possui um custo computacional elevado, o treinamento foi realizado previamente. Nesta etapa, o arquivo do modelo é baixado por meio do Google Drive. O treinamento permanece comentado para evitar uma nova execução desnecessária. Após o download, `keras.models.load_model()` restaura o modelo completo, incluindo a arquitetura e os parâmetros obtidos durante o *fine-tuning*. A partir desse ponto, o modelo pode ser utilizado diretamente para avaliação, classificação de novas resenhas e análise das representações latentes. Para realizar o treinamento novamente, descomente o bloco de treinamento e comente a etapa de carregamento do modelo previamente salvo.

In [20]:
# baixe os pesos do bert treinado com fine-tuning pelo google drive
%pip install -q gdown

import gdown

FILE_ID = "17Uk7TvZSc1-Kaf-4aRy6MEJfNyQ5Lyx4"
DATASET_ZIP = str(OUTPUT_DIR / "bert_best.keras")

gdown.download(
    id=FILE_ID,
    output=DATASET_ZIP
)

In [ ]:
# history = model.fit(
#     train_dataset,
#     validation_data=val_dataset,
#     epochs=NUM_TRAIN_EPOCHS,
#     callbacks=[
#         checkpoint_callback,
#         early_stopping_callback,
#         tensorboard_callback
#     ]
# )

# carrega o modelo previamente salvo
model = keras.models.load_model(
    str(OUTPUT_DIR / "bert_best.keras")
)

Epoch 1/4
704/704 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.6834 - loss: 0.7443

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 5:</b><br>
  Qual é a finalidade do warmup da taxa de aprendizado no início do treinamento? Por que essa estratégia pode ser particularmente útil durante o fine-tuning de um modelo pré-treinado?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

## Análise do Modelo

Após o treinamento e a recuperação dos pesos do modelo, esta etapa é dedicada à análise de seu desempenho e de suas representações aprendidas. Inicialmente, são avaliadas as predições realizadas sobre o conjunto de teste, utilizando métricas e a matriz de confusão para verificar a capacidade do modelo em distinguir corretamente as classes de predição negativa e positiva.

Em seguida, são analisadas as representações obtidas pelo BERT no espaço latente, utilizando redução de dimensionalidade por meio do t-SNE, de modo a permitir visualizar como as amostras das diferentes classes se distribuem nesse espaço e verificar se o modelo desenvolveu representações capazes de separar semanticamente os sentimentos.

In [ ]:
# cria o dataset de teste
test_dataset = tf.data.Dataset.from_tensor_slices(
    (test_x, test_labels)
)

test_dataset = test_dataset.batch(
    BATCH_SIZE,
    drop_remainder=False
)

test_dataset = test_dataset.prefetch(
    tf.data.AUTOTUNE
)

In [ ]:
results = model.evaluate(
    test_dataset,
    return_dict=True
)

print(results)

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 6:</b><br>
  Compare a acurácia obtida no conjunto de teste com o desempenho observado durante o treinamento e a validação. O resultado sugere boa capacidade de generalização? Justifique.
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

In [ ]:
# retorna lista de (frase, probabilidades, rótulo) para cada entrada
def get_prediction(in_sentences):
    labels = ["Negative", "Positive"]

    # tokeniza as frases
    features = tokenizer(
        in_sentences,
        max_length=MAX_SEQ_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="np"
    )

    # converte para o formato esperado pelo keras-hub
    predict_x = {
        "token_ids": features["input_ids"],
        "segment_ids": features["token_type_ids"],
        "padding_mask": features["attention_mask"].astype(bool)
    }

    # faz as previsões e converte logits em probabilidades
    logits = model.predict(predict_x, verbose=0)
    probabilities = keras.ops.softmax(logits, axis=-1)
    predicted_labels = keras.ops.argmax(probabilities, axis=-1)

    # converte para numpy
    probabilities = keras.ops.convert_to_numpy(probabilities)
    predicted_labels = keras.ops.convert_to_numpy(predicted_labels)

    return [
        (sentence, probabilities[i], labels[predicted_labels[i]])
        for i, sentence in enumerate(in_sentences)
    ]

In [ ]:
my_reviews = [
    "This movie was absolutely fantastic, the acting was great and the plot was amazing.",
    "I hated this film. It was boring, slow, and a complete waste of time.",
    "The visual effects were incredible and the sound design was top tier, but the story was empty and boring.",
    "I really wanted to love this movie. The trailer looked like a masterpiece, but the actual film was a huge letdown.",
    "If you are a fan of mindless action with zero plot, this is perfect. Otherwise, stay away.",
    "Oh, brilliant. Another superhero movie with the exact same plot as the last fifty ones. Just what we needed.",
    "It was not the best movie I have ever seen, but it was certainly not the worst. It was just okay.",
    "I cannot say I didn't enjoy it. It was actually better than I expected.",
    "At first I thought this movie was going to be terrible, but the ending was absolutely perfect.",
    "Funny at times, yet I wouldn't watch it again.",
    "Nice visuals and great performances, but the plot was completely confusing, waste of time and terrible."
]

predictions = get_prediction(my_reviews)

In [ ]:
# visualiza os resultados da predição
predictions

In [ ]:
# faz as predições no conjunto de teste
logits = model.predict(test_x, verbose=1)

# converte logits em classes preditas
predictions = np.argmax(logits, axis=1)

# gera a matriz de confusão
cm = confusion_matrix(test_labels, predictions)

# plota a matriz de confusão
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Negative", "Positive"],
    yticklabels=["Negative", "Positive"],
    cbar=False
)
plt.title("Matriz de Confusão (Dados de Teste)", fontsize=14, fontweight="bold", pad=15)
plt.ylabel("Classe Verdadeira", fontsize=12)
plt.xlabel("Classe Predita", fontsize=12)
plt.tight_layout()
plt.show()

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 7:</b><br>
  Se o conjunto de teste tivesse uma quantidade muito maior de críticas positivas do que negativas, a acurácia ainda seria uma métrica suficiente para avaliar o modelo? Por quê?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

In [ ]:
# modelo auxiliar para extrair o pooled output do bert
feature_extractor = keras.Model(
    inputs=model.inputs,
    outputs=pooled_output
)

# seleciona subconjunto do teste para visualização
num_samples_to_visualize = 1000
visualization_x = {
    "token_ids": test_features["input_ids"][:num_samples_to_visualize],
    "segment_ids": test_features["token_type_ids"][:num_samples_to_visualize],
    "padding_mask": test_features["attention_mask"][:num_samples_to_visualize].astype(bool)
}
labels_array = np.array(test_labels[:num_samples_to_visualize])

# extrai as representações do espaço latente
latent_matrix = feature_extractor.predict(visualization_x, verbose=1)
print(f"shape da matriz latente: {latent_matrix.shape}")
print(f"shape dos rótulos: {labels_array.shape}")

# redução de dimensionalidade com t-sne
tsne = TSNE(n_components=2, random_state=42, perplexity=30.0, max_iter=1000)
tsne_results = tsne.fit_transform(latent_matrix)

# plota o espaço latente com cores por sentimento
sns.set_theme(style="whitegrid", palette="muted")
plt.figure(figsize=(10, 8))
scatter_plot = sns.scatterplot(
    x=tsne_results[:, 0],
    y=tsne_results[:, 1],
    hue=labels_array,
    palette=["#e74c3c", "#3498db"],
    alpha=0.75,
    edgecolor="k",
    s=60
)
legend_handles, _ = scatter_plot.get_legend_handles_labels()
plt.legend(handles=legend_handles, labels=["Negative (0)", "Positive (1)"], title="Sentimento", loc="best", fontsize=11)
plt.title("Visualização t-SNE do Espaço Latente (BERT)", fontsize=16, fontweight="bold", pad=15)
plt.xlabel("Componente t-SNE 1", fontsize=12)
plt.ylabel("Componente t-SNE 2", fontsize=12)
plt.tight_layout()
plt.show()

<div style="
    background-color:#f9f9f9;
    border-left:6px solid #4A90E2;
    padding:15px;
    border-radius:8px;
    margin:10px 0;
">
  <b style="color:#2c3e50;"> Pergunta 8:</b><br>
  Observe a visualização t-SNE das representações latentes. As amostras positivas e negativas formam agrupamentos bem separados? O que essa distribuição sugere sobre as representações aprendidas pelo BERT?
</div>

<em> Adicione sua resposta abaixo desta célula.</em>

## Avaliação dos Modelos

A partir das matrizes de confusão obtidas no conjunto de teste (25.000 amostras balanceadas, com 12.500 positivas e 12.500 negativas), apresentamos as principais métricas de avaliação para cada arquitetura testada ao longo deste experimento.

**Observação**: Devido à natureza estocástica dos treinamentos realizados, como a inicialização aleatória de pesos e a amostragem de *batches*, os resultados quantitativos obtidos de cada experimento estão sujeitos a pequenas variações a cada nova execução.

| Arquitetura | Accuracy | Precision (Macro) | Recall (Macro) | F1-Score (Macro) |
| :--- | :---: | :---: | :---: | :---: |
| **RNN (Baseline)** | 0.8142 | 0.8144 | 0.8142 | 0.8141 |
| **LSTM** | 0.8524 | 0.8535 | 0.8524 | 0.8521 |
| **Transformer (Simples)** | 0.8691 | 0.8697 | 0.8691 | 0.8690 |
| **BERT (Fine-tuning)** | **0.918480** | **0.918493** | **0.918480** | **0.918479** |

O salto mais expressivo em capacidade de generalização ocorreu na transição da rede RNN padrão para a LSTM, evidenciando a eficácia das *gates* em mitigar o desvanecimento do gradiente e capturar dependências de longo prazo essenciais em textos como reviews de filmes.
A LSTM obteve um ganho de *precision* (a partir da redução dos falsos positivos), mas teve penalidade o *recall*, tornando-se um modelo mais conservador ao classificar *reviews* positivos.
  
A introdução do mecanismo de *self-attention* no Transformer simples estabilizou o *trade-off* entre *precision* e *recall*, superando a LSTM no *F1-score*.
